In [ ]:
import pandas as pd
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import grad
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
from joblib import load

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

In [ ]:
class Swish(nn.Module):
    def __init__(self):
        super(Swish, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

class NeuralNetwork(nn.Module):
    def __init__(self, width = 32, use_batch_norm=False):
        super(NeuralNetwork, self).__init__()
        self.use_batch_norm = use_batch_norm
        self.activation = Swish()
        self.fc1 = nn.Linear(6, width)
        self.bn1 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc2 = nn.Linear(width, width)
        self.bn2 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc3 = nn.Linear(width, width)
        self.bn3 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc4 = nn.Linear(width, width)
        self.bn4 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc5 = nn.Linear(width, 1)

    def forward(self, x):
        x = self.activation(self.bn1(self.fc1(x)))
        x = self.activation(self.bn2(self.fc2(x)))
        x = self.activation(self.bn3(self.fc3(x)))
        x = self.activation(self.bn4(self.fc4(x)))
        x = self.fc5(x)
        return x

bergominn = NeuralNetwork(256, True).to(device)

model_path = "bergominn.pth"
state_dict = torch.load(model_path)

bergominn.load_state_dict(state_dict)

scaler_path = 'rbergomi_scaler.bin'
standard_scaler = load(scaler_path)

In [ ]:
from py_vollib.black_scholes.implied_volatility import implied_volatility
import rbergomi

def rBergomiIV(S0, K, T, rho, eta, H, xi_0, num_paths, num_steps):
    a = H - 0.5
    rB = rbergomi.rBergomi(n = num_steps, N = num_paths, T = T, a = a)
    dW1 = rB.dW1()
    dW2 = rB.dW2()
    Y = rB.Y(dW1)
    dB = rB.dB(dW1, dW2, rho = rho)
    V = rB.V(Y, xi = xi_0, eta = eta)
    S = rB.S(V, dB)
    ST = S[:,-1][:,np.newaxis]
    call_payoff = np.maximum(ST - K,0)
    call_price = np.mean(call_payoff, axis = 0)[:,np.newaxis].item()
    iv = implied_volatility(call_price, S0, K, T, 0.0, 'c')
    return iv

In [ ]:
mats = np.round(np.linspace(0.1,5,8),1)
mats

strikes = np.round(np.linspace(0.7,1.3,11),1)
strikes

In [ ]:
S0 = 1.0
eta = 2.5
rho = -0.5
H = 0.25
xi_0 = 0.075

In [ ]:
bergomi_smile_lst = list()

for imat in mats:
    smile_lst = list()
    for ik in strikes:
        iv = rBergomiIV(S0, ik, imat, rho, eta, H, xi_0, 50000, 100)
        smile_lst.append(iv)
    bergomi_smile_lst.append(smile_lst)

bergomi_smile = np.array(bergomi_smile_lst)

In [ ]:
bergominn_smile_lst = list()

for imat in mats:
    for ik in strikes:
        input_lst = list()
        input_lst.append(eta)
        input_lst.append(rho)
        input_lst.append(H)
        input_lst.append(xi_0)
        input_lst.append(imat)
        input_lst.append(ik)
        bergominn_smile_lst.append(input_lst)

heading_list = ['eta', 'rho', 'H', 'xi_0', 'tau', 'K']
df = pd.DataFrame(bergominn_smile_lst, columns=heading_list)
df.head()

bergomi_X = standard_scaler.transform(df)

tbergomi_X = torch.Tensor(bergomi_X).to(device)

bergominn_iv = bergominn(tbergomi_X)

bergominn_iv = bergominn_iv.reshape((8, 11)).detach().cpu().numpy()

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(14, 10))
axs = axs.flatten()

for i, maturity in enumerate(mats):
    axs[i].plot(strikes, bergomi_smile[i], color='black', linestyle = '-', label='rBergomi')
    axs[i].plot(strikes, bergominn_iv[i], color='black', linestyle = '--', label='DNN')
    axs[i].set_title(f'Maturity = {maturity} years')
    axs[i].set_xlabel('Relative Strike (K/S)')
    axs[i].set_ylabel('Implied Volatility')
    axs[i].legend()